In [2]:
from pathlib import Path

import pandas as pd

from IPython.display import display
from sklearn.model_selection import train_test_split


# Buscar el corpus según la ubicación del notebook actual
rutas_posibles = [
    Path("corpus_normalizado.pkl"),
    Path("../Lab1/corpus_normalizado.pkl"),
    Path("Lab1/corpus_normalizado.pkl")
]

ruta_corpus = next(
    (
        ruta
        for ruta in rutas_posibles
        if ruta.exists()
    ),
    None
)

if ruta_corpus is None:
    raise FileNotFoundError(
        "No se encontró 'corpus_normalizado.pkl'."
    )

corpus = pd.read_pickle(
    ruta_corpus
)

columnas_requeridas = [
    "url",
    "news",
    "Type",
    "tokens_lematizados",
    "News_normalizada"
]

columnas_faltantes = [
    columna
    for columna in columnas_requeridas
    if columna not in corpus.columns
]

if columnas_faltantes:
    raise ValueError(
        f"Faltan columnas necesarias: {columnas_faltantes}"
    )

semilla_aleatoria = 42

# Agrupar documentos con el mismo texto normalizado
clave_texto = (
    corpus["News_normalizada"]
    .fillna("")
    .str.strip()
)

tabla_grupos_texto = (
    corpus
    .assign(_clave_texto=clave_texto)
    .groupby(
        "_clave_texto",
        sort=False
    )
    .agg(
        Categoria_estratificacion=(
            "Type",
            lambda categorias:
                categorias.mode().iloc[0]
        ),
        Cantidad_documentos=(
            "Type",
            "size"
        )
    )
    .reset_index()
)

# Separar los grupos en entrenamiento y conjunto temporal
grupos_entrenamiento, grupos_temporales = (
    train_test_split(
        tabla_grupos_texto,
        test_size=0.30,
        random_state=semilla_aleatoria,
        stratify=tabla_grupos_texto[
            "Categoria_estratificacion"
        ]
    )
)

# Dividir el conjunto temporal entre validación y prueba
grupos_validacion, grupos_prueba = (
    train_test_split(
        grupos_temporales,
        test_size=0.50,
        random_state=semilla_aleatoria,
        stratify=grupos_temporales[
            "Categoria_estratificacion"
        ]
    )
)

claves_entrenamiento = set(
    grupos_entrenamiento["_clave_texto"]
)

claves_validacion = set(
    grupos_validacion["_clave_texto"]
)

claves_prueba = set(
    grupos_prueba["_clave_texto"]
)

corpus_entrenamiento = corpus.loc[
    clave_texto.isin(
        claves_entrenamiento
    )
].copy()

corpus_validacion = corpus.loc[
    clave_texto.isin(
        claves_validacion
    )
].copy()

corpus_prueba = corpus.loc[
    clave_texto.isin(
        claves_prueba
    )
].copy()

intersecciones_texto = {
    "Entrenamiento - Validación": len(
        claves_entrenamiento
        & claves_validacion
    ),
    "Entrenamiento - Prueba": len(
        claves_entrenamiento
        & claves_prueba
    ),
    "Validación - Prueba": len(
        claves_validacion
        & claves_prueba
    )
}

if (
    len(corpus_entrenamiento)
    + len(corpus_validacion)
    + len(corpus_prueba)
    != len(corpus)
):
    raise AssertionError(
        "Las particiones no contienen todos los documentos."
    )

if not all(
    cantidad == 0
    for cantidad in intersecciones_texto.values()
):
    raise AssertionError(
        "Existen textos normalizados compartidos "
        "entre las particiones."
    )

resumen_particiones = pd.DataFrame({
    "Conjunto": [
        "Entrenamiento",
        "Validación",
        "Prueba"
    ],
    "Cantidad de documentos": [
        len(corpus_entrenamiento),
        len(corpus_validacion),
        len(corpus_prueba)
    ]
})

resumen_particiones["Porcentaje del corpus"] = (
    resumen_particiones[
        "Cantidad de documentos"
    ]
    / len(corpus)
    * 100
).round(2)

print(
    f"Corpus cargado desde: {ruta_corpus.resolve()}"
)

display(
    resumen_particiones
)

print("Textos normalizados compartidos entre conjuntos:")

for comparacion, cantidad in (
    intersecciones_texto.items()
):
    print(f"{comparacion}: {cantidad}")

Corpus cargado desde: /Users/josetanchez/Desktop/NLP-Labs-JoseTanchez/Lab1/corpus_normalizado.pkl


,Conjunto,Cantidad de documentos,Porcentaje del corpus
0,Entrenamiento,850,69.84
1,Validación,185,15.20
2,Prueba,182,14.95


Textos normalizados compartidos entre conjuntos:
Entrenamiento - Validación: 0
Entrenamiento - Prueba: 0
Validación - Prueba: 0


In [3]:
from sklearn.feature_extraction.text import (
    CountVectorizer,
    TfidfVectorizer
)


textos_entrenamiento = (
    corpus_entrenamiento[
        "News_normalizada"
    ]
    .fillna("")
)

textos_validacion = (
    corpus_validacion[
        "News_normalizada"
    ]
    .fillna("")
)

textos_prueba = (
    corpus_prueba[
        "News_normalizada"
    ]
    .fillna("")
)

etiquetas_entrenamiento = (
    corpus_entrenamiento["Type"]
)

etiquetas_validacion = (
    corpus_validacion["Type"]
)

etiquetas_prueba = (
    corpus_prueba["Type"]
)

orden_categorias = (
    corpus["Type"]
    .value_counts()
    .index
    .tolist()
)

# Bolsa de Palabras
vectorizador_bow = CountVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False
)

matriz_bow_entrenamiento = (
    vectorizador_bow.fit_transform(
        textos_entrenamiento
    )
)

matriz_bow_validacion = (
    vectorizador_bow.transform(
        textos_validacion
    )
)

matriz_bow_prueba = (
    vectorizador_bow.transform(
        textos_prueba
    )
)

vocabulario_bow = (
    vectorizador_bow.get_feature_names_out()
)

# TF-IDF
vectorizador_tfidf = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False
)

matriz_tfidf_entrenamiento = (
    vectorizador_tfidf.fit_transform(
        textos_entrenamiento
    )
)

matriz_tfidf_validacion = (
    vectorizador_tfidf.transform(
        textos_validacion
    )
)

matriz_tfidf_prueba = (
    vectorizador_tfidf.transform(
        textos_prueba
    )
)

vocabulario_tfidf = (
    vectorizador_tfidf.get_feature_names_out()
)

if not (
    matriz_bow_entrenamiento.shape[0]
    == len(etiquetas_entrenamiento)
    and matriz_bow_validacion.shape[0]
    == len(etiquetas_validacion)
    and matriz_bow_prueba.shape[0]
    == len(etiquetas_prueba)
):
    raise AssertionError(
        "Las matrices BoW no coinciden con sus etiquetas."
    )

if not (
    matriz_tfidf_entrenamiento.shape[0]
    == len(etiquetas_entrenamiento)
    and matriz_tfidf_validacion.shape[0]
    == len(etiquetas_validacion)
    and matriz_tfidf_prueba.shape[0]
    == len(etiquetas_prueba)
):
    raise AssertionError(
        "Las matrices TF-IDF no coinciden con sus etiquetas."
    )

if not (
    matriz_bow_entrenamiento.shape[1]
    == matriz_bow_validacion.shape[1]
    == matriz_bow_prueba.shape[1]
    == len(vocabulario_bow)
):
    raise AssertionError(
        "Las matrices BoW no comparten el mismo vocabulario."
    )

if not (
    matriz_tfidf_entrenamiento.shape[1]
    == matriz_tfidf_validacion.shape[1]
    == matriz_tfidf_prueba.shape[1]
    == len(vocabulario_tfidf)
):
    raise AssertionError(
        "Las matrices TF-IDF no comparten "
        "el mismo vocabulario."
    )

resumen_representaciones = pd.DataFrame({
    "Representación": [
        "BoW",
        "TF-IDF"
    ],
    "Entrenamiento": [
        str(matriz_bow_entrenamiento.shape),
        str(matriz_tfidf_entrenamiento.shape)
    ],
    "Validación": [
        str(matriz_bow_validacion.shape),
        str(matriz_tfidf_validacion.shape)
    ],
    "Prueba": [
        str(matriz_bow_prueba.shape),
        str(matriz_tfidf_prueba.shape)
    ],
    "Tamaño del vocabulario": [
        len(vocabulario_bow),
        len(vocabulario_tfidf)
    ]
})

display(
    resumen_representaciones
)

,Representación,Entrenamiento,Validación,Prueba,Tamaño del vocabulario
0,BoW,"(850, 17922)","(185, 17922)","(182, 17922)",17922
1,TF-IDF,"(850, 17922)","(185, 17922)","(182, 17922)",17922


In [4]:
from sklearn.linear_model import LogisticRegression


modelo_regresion_logistica_bow = (
    LogisticRegression(
        solver="lbfgs",
        max_iter=1000,
        random_state=semilla_aleatoria
    )
)

modelo_regresion_logistica_bow.fit(
    matriz_bow_entrenamiento,
    etiquetas_entrenamiento
)

print(
    "Modelo de Regresión Logística "
    "con BoW entrenado correctamente."
)

print(
    "Cantidad de categorías aprendidas:",
    len(
        modelo_regresion_logistica_bow.classes_
    )
)

print("Categorías aprendidas:")

print(
    modelo_regresion_logistica_bow.classes_
)

Modelo de Regresión Logística con BoW entrenado correctamente.
Cantidad de categorías aprendidas: 7
Categorías aprendidas:
['Alianzas' 'Innovacion' 'Macroeconomia' 'Otra' 'Regulaciones'
 'Reputacion' 'Sostenibilidad']


In [5]:
modelo_regresion_logistica_tfidf = (
    LogisticRegression(
        solver="lbfgs",
        max_iter=1000,
        random_state=semilla_aleatoria
    )
)

modelo_regresion_logistica_tfidf.fit(
    matriz_tfidf_entrenamiento,
    etiquetas_entrenamiento
)

print(
    "Modelo de Regresión Logística "
    "con TF-IDF entrenado correctamente."
)

print(
    "Cantidad de categorías aprendidas:",
    len(
        modelo_regresion_logistica_tfidf.classes_
    )
)

print("Categorías aprendidas:")

print(
    modelo_regresion_logistica_tfidf.classes_
)

Modelo de Regresión Logística con TF-IDF entrenado correctamente.
Cantidad de categorías aprendidas: 7
Categorías aprendidas:
['Alianzas' 'Innovacion' 'Macroeconomia' 'Otra' 'Regulaciones'
 'Reputacion' 'Sostenibilidad']


In [6]:
verificaciones_clasificadores = {
    "Ambos modelos aprendieron las mismas categorías": (
        modelo_regresion_logistica_bow
        .classes_
        .tolist()
        == modelo_regresion_logistica_tfidf
        .classes_
        .tolist()
    ),
    "BoW contiene una fila de pesos por categoría": (
        modelo_regresion_logistica_bow
        .coef_
        .shape[0]
        == len(
            modelo_regresion_logistica_bow
            .classes_
        )
    ),
    "TF-IDF contiene una fila de pesos por categoría": (
        modelo_regresion_logistica_tfidf
        .coef_
        .shape[0]
        == len(
            modelo_regresion_logistica_tfidf
            .classes_
        )
    ),
    "Los pesos de BoW coinciden con su vocabulario": (
        modelo_regresion_logistica_bow
        .coef_
        .shape[1]
        == len(vocabulario_bow)
    ),
    "Los pesos de TF-IDF coinciden con su vocabulario": (
        modelo_regresion_logistica_tfidf
        .coef_
        .shape[1]
        == len(vocabulario_tfidf)
    ),
    "BoW convergió antes del límite de iteraciones": (
        modelo_regresion_logistica_bow
        .n_iter_
        .max()
        < modelo_regresion_logistica_bow
        .max_iter
    ),
    "TF-IDF convergió antes del límite de iteraciones": (
        modelo_regresion_logistica_tfidf
        .n_iter_
        .max()
        < modelo_regresion_logistica_tfidf
        .max_iter
    )
}

tabla_verificacion_clasificadores = pd.DataFrame({
    "Verificación":
        verificaciones_clasificadores.keys(),
    "Resultado": [
        "Correcto" if resultado else "Error"
        for resultado
        in verificaciones_clasificadores.values()
    ]
})

if not all(
    verificaciones_clasificadores.values()
):
    raise AssertionError(
        "Una o más verificaciones de los "
        "clasificadores fallaron."
    )

resumen_clasificadores = pd.DataFrame({
    "Modelo": [
        "Regresión Logística + BoW",
        "Regresión Logística + TF-IDF"
    ],
    "Categorías": [
        len(
            modelo_regresion_logistica_bow
            .classes_
        ),
        len(
            modelo_regresion_logistica_tfidf
            .classes_
        )
    ],
    "Forma de coef_": [
        str(
            modelo_regresion_logistica_bow
            .coef_
            .shape
        ),
        str(
            modelo_regresion_logistica_tfidf
            .coef_
            .shape
        )
    ],
    "Forma de intercept_": [
        str(
            modelo_regresion_logistica_bow
            .intercept_
            .shape
        ),
        str(
            modelo_regresion_logistica_tfidf
            .intercept_
            .shape
        )
    ],
    "Iteraciones": [
        int(
            modelo_regresion_logistica_bow
            .n_iter_
            .max()
        ),
        int(
            modelo_regresion_logistica_tfidf
            .n_iter_
            .max()
        )
    ]
})

display(
    resumen_clasificadores
)

display(
    tabla_verificacion_clasificadores
)

print(
    "Los dos clasificadores se construyeron "
    "y verificaron correctamente."
)

,Modelo,Categorías,Forma de coef_,Forma de intercept_,Iteraciones
0,Regresión Logística + BoW,7,"(7, 17922)","(7,)",66
1,Regresión Logística + TF-IDF,7,"(7, 17922)","(7,)",31


,Verificación,Resultado
0,Ambos modelos aprendieron las mismas categorías,Correcto
1,BoW contiene una fila de pesos por categoría,Correcto
2,TF-IDF contiene una fila de pesos por categoría,Correcto
3,Los pesos de BoW coinciden con su vocabulario,Correcto
4,Los pesos de TF-IDF coinciden con su vocabulario,Correcto
5,BoW convergió antes del límite de iteraciones,Correcto
6,TF-IDF convergió antes del límite de iteraciones,Correcto


Los dos clasificadores se construyeron y verificaron correctamente.


# 1. Construcción del clasificador de Regresión Logística

Se construyeron dos clasificadores de Regresión Logística para predecir la categoría de cada noticia:

1. Regresión Logística con Bolsa de Palabras.
2. Regresión Logística con TF-IDF.

Se conservaron el corpus, las particiones y la configuración de los vectorizadores utilizados en el Laboratorio 3. De esta forma, los clasificadores lineales y los modelos de Naive Bayes utilizan los mismos documentos y espacios de características.

## 1.1 Preparación de los datos

El corpus contiene **1,217 documentos**, distribuidos mediante una división estratificada y reproducible con `random_state=42`.

| Conjunto | Documentos | Porcentaje |
|---|---:|---:|
| Entrenamiento | 850 | 69.84 % |
| Validación | 185 | 15.20 % |
| Prueba | 182 | 14.95 % |

Los documentos con el mismo texto normalizado se mantuvieron dentro de una sola partición. Las intersecciones entre entrenamiento, validación y prueba fueron iguales a cero, por lo que no existen textos normalizados compartidos entre conjuntos.

Los vectorizadores se ajustaron exclusivamente con los documentos de entrenamiento. Los conjuntos de validación y prueba se transformaron sin modificar el vocabulario aprendido.

| Representación | Entrenamiento | Validación | Prueba | Vocabulario |
|---|---|---|---|---:|
| BoW | (850, 17,922) | (185, 17,922) | (182, 17,922) | 17,922 |
| TF-IDF | (850, 17,922) | (185, 17,922) | (182, 17,922) | 17,922 |

## 1.2 Regresión Logística con BoW

El primer clasificador se entrenó con `matriz_bow_entrenamiento`, que representa cada noticia mediante los conteos de sus palabras.

Se utilizó `LogisticRegression` con el optimizador `lbfgs` y un máximo de 1,000 iteraciones. El modelo aprendió las siete categorías del corpus:

- `Alianzas`
- `Innovacion`
- `Macroeconomia`
- `Otra`
- `Regulaciones`
- `Reputacion`
- `Sostenibilidad`

El entrenamiento convergió después de **66 iteraciones**. La matriz `coef_` tiene forma **(7, 17,922)**, correspondiente a una fila de pesos por categoría y una columna por característica. El vector `intercept_` tiene forma **(7,)**.

## 1.3 Regresión Logística con TF-IDF

El segundo clasificador se entrenó con `matriz_tfidf_entrenamiento`. En esta representación, las palabras reciben valores de acuerdo con su importancia dentro de cada documento y su frecuencia en el conjunto de entrenamiento.

Se utilizó la misma configuración de Regresión Logística. El modelo también aprendió las siete categorías y convergió después de **31 iteraciones**.

Su matriz `coef_` tiene forma **(7, 17,922)** y su vector `intercept_` tiene forma **(7,)**. Por lo tanto, cada categoría cuenta con un peso para cada término del vocabulario y con su propio intercepto.

## 1.4 Interpretación del modelo lineal

La operación fundamental de un clasificador lineal puede expresarse como:

\[
z = \mathbf{w}\cdot\mathbf{x} + b
\]

En el contexto de la clasificación de noticias:

- \(\mathbf{x}\) representa las características de una noticia. Sus componentes son conteos de palabras en BoW o valores TF-IDF.
- \(\mathbf{w}\) representa los pesos aprendidos para las características.
- \(b\) representa el intercepto aprendido por el modelo.
- \(z\) representa el puntaje lineal producido para una categoría.

Un peso positivo aumenta el puntaje de una categoría cuando la característica aparece en el documento, mientras que un peso negativo lo reduce.

En un problema binario, la función sigmoide permite transformar un puntaje lineal en un valor entre cero y uno:

\[
\sigma(z) = \frac{1}{1 + e^{-z}}
\]

Este valor puede interpretarse como la probabilidad estimada de la clase positiva. Posteriormente, la probabilidad se utiliza para obtener una predicción.

En este corpus existen siete categorías, por lo que el problema es multiclase. La configuración utilizada con `lbfgs` aplica una estrategia multinomial: se calcula un puntaje para cada categoría \(c\):

\[
z_c = \mathbf{w}_c\cdot\mathbf{x} + b_c
\]

Los siete puntajes se normalizan conjuntamente mediante la función softmax:

\[
P(y=c\mid\mathbf{x})
=
\frac{e^{z_c}}
{\sum_j e^{z_j}}
\]

De esta manera, las probabilidades de todas las categorías suman uno. La predicción final corresponde a la categoría con la probabilidad más alta:

\[
\hat{y}
=
\underset{c}{\operatorname{argmax}}
\ P(y=c\mid\mathbf{x})
\]

La sigmoide describe el caso binario fundamental, mientras que softmax constituye su extensión coherente para la clasificación multinomial utilizada en este laboratorio.

## 1.5 Verificación de los clasificadores

Las verificaciones confirmaron que:

1. Ambos modelos aprendieron exactamente las mismas siete categorías.
2. Cada categoría posee su propia fila de coeficientes.
3. Los coeficientes de cada modelo coinciden con las 17,922 características de su vectorizador.
4. Cada modelo posee siete interceptos.
5. Ambos entrenamientos convergieron antes del límite de 1,000 iteraciones.
6. Los modelos fueron entrenados únicamente con los 850 documentos del conjunto de entrenamiento.

Por lo tanto, los clasificadores de Regresión Logística con BoW y TF-IDF quedaron construidos correctamente y preparados para su evaluación posterior.